# ChurnGuard — Exploratory Data Analysis
Goal: understand the dataset before we build anything. No cleaning pipelines here — just observations that will inform our later decisions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the raw dataset from the data/raw folder
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# How many rows and columns do we have?
print('Shape:', df.shape)
print()

# What type is each column? (important — wrong types = bugs later)
print('Column types:')
print(df.dtypes)
print()

# Preview the first 5 rows
df.head()

## Target Distribution
How many customers churned vs stayed? We need to know if the dataset is imbalanced.

In [ ]:
# Count of each class (No = stayed, Yes = churned)
churn_counts = df['Churn'].value_counts()
print('Raw counts:')
print(churn_counts)
print()

# Percentage breakdown — this tells us how imbalanced the classes are
print('Percentage breakdown:')
print(df['Churn'].value_counts(normalize=True).round(3) * 100)

# Bar chart of churn distribution
churn_counts.plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Churn Distribution')
plt.xlabel('Churn')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Key insight: ~26.5% churned — imbalanced but not extreme.
# A naive model predicting 'No churn' always would get 73.5% accuracy but catch 0 churners.
# We'll handle this in Phase 2 with class weighting or SMOTE.

## Missing Values
Are there any nulls? Any columns that look numeric but are stored as strings (a common CSV issue)?

In [ ]:
# Check for actual null values across all columns
print('Null counts per column:')
print(df.isnull().sum())
print()

# TotalCharges is stored as a string (object) but should be float
# This usually means some rows have blank strings instead of numbers
blank_total_charges = df[df['TotalCharges'] == ' '].shape[0]
print(f'Rows with blank TotalCharges: {blank_total_charges}')

# Key insight: TotalCharges has 11 blank rows — these are likely new customers
# with tenure=0 who haven't been billed yet. We'll handle this in preprocessing.

## Tenure Distribution
How long have customers been with the company? Tenure is usually one of the strongest churn signals.

In [ ]:
# Plot tenure distribution split by churn status
# This shows whether short-tenure customers churn more
plt.figure(figsize=(10, 4))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, palette=['steelblue', 'salmon'])
plt.title('Tenure Distribution by Churn')
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Summary stats for tenure
print('Tenure stats by churn status:')
print(df.groupby('Churn')['tenure'].describe().round(1))

# Key insight: churned customers tend to have much lower tenure
# — they leave early or not at all. Tenure is a strong feature.

## Monthly Charges Distribution
Do higher-paying customers churn more?

In [ ]:
# Distribution of monthly charges split by churn
plt.figure(figsize=(10, 4))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', bins=30, palette=['steelblue', 'salmon'])
plt.title('Monthly Charges Distribution by Churn')
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Summary stats
print('Monthly charges stats by churn status:')
print(df.groupby('Churn')['MonthlyCharges'].describe().round(1))

# Key insight: churned customers tend to pay more per month
# — possibly on expensive month-to-month contracts.

## Contract Type vs Churn
Contract type is likely one of the strongest predictors — customers on month-to-month contracts have no lock-in.

In [ ]:
# Churn rate by contract type
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack()
print('Churn rate by contract type:')
print(contract_churn.round(3) * 100)
print()

# Grouped bar chart
contract_churn.plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.legend(['No Churn', 'Churn'])
plt.tight_layout()
plt.show()

# Key insight: month-to-month customers churn at a dramatically higher rate
# — contract type will be one of the top features in our model.

## Correlation Heatmap (Numeric Features)
Which numeric features move together? And which correlate with churn?

In [ ]:
# Convert Churn to numeric (1 = churned, 0 = stayed) for correlation
df['Churn_numeric'] = (df['Churn'] == 'Yes').astype(int)

# Convert TotalCharges to numeric — coerce blanks to NaN
df['TotalCharges_numeric'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Select numeric columns for correlation
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges_numeric', 'Churn_numeric']

# Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr().round(2), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

# Key insight: tenure negatively correlates with churn (longer = less likely to leave)
# MonthlyCharges positively correlates with churn (higher bill = more likely to leave)
# TotalCharges is highly correlated with tenure — it's not adding much new info.

## EDA Summary
Key findings that will drive our modeling decisions:

In [ ]:
print("""
EDA SUMMARY
===========
1. Dataset: 7,043 customers, 21 columns
2. Class imbalance: 73.5% stayed, 26.5% churned
   → Will use class_weight='balanced' or SMOTE in Phase 2
3. TotalCharges: stored as string, 11 blank rows (new customers)
   → Will convert to float, fill blanks with 0 or drop in preprocessing
4. Strong churn signals:
   - Low tenure (new customers churn more)
   - High monthly charges
   - Month-to-month contract (no lock-in)
5. TotalCharges is redundant with tenure — may drop in feature engineering
""")